## GlycoAID – Diabetes Risk Prediction System

### A Supervised Learning (Binary Classification) problem

#### Training Model

In [ ]:
# Import Libraries & Tools

import pandas as pd # pandas  → reads and organizes data like an Excel sheet

import joblib # joblib  → saves our trained model to a file

from sklearn.model_selection import train_test_split # splits data into study vs. test piles

from sklearn.linear_model import LogisticRegression # Model 1: Logistic Regression

from sklearn.tree import DecisionTreeClassifier # Model 2: Decision Tree

from sklearn.metrics import accuracy_score # how often the model gets it right (out of 100%)

from sklearn.preprocessing import StandardScaler #makes all numbers use the same "scale"

from sklearn.pipeline import Pipeline 

### Load the dataset

In [21]:
# Loading the dataset directly from the internet
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
data = pd.read_csv(url) # pd.read_csv reads the online table and stores it in 'df'

In [23]:
print(data.head())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


### Exploratory Data Analysis

In [25]:
# Summary Statistics (min, max, average, etc.)
print(data.describe())


       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  
min      0.000000                  

### Fix wrong zero values

In [30]:
# Some columns Glucose, BMI, and Insulin contain ZEROS.
columns_with_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

for col in columns_with_zero:
    # Count how many zeros exist in this column before fixing
    zero_count = (data[col] == 0).sum()

# Replace all 0s in this column with the median of the non-zero values
for col in columns_with_zero:
    data[col] = data[col].replace(0, data[col].mean()) # COME BACK TO TEST WITH THE MEDIAN


### Separate inputs and answers

In [32]:
X = data.drop("Outcome", axis=1)  # symptoms
y = data["Outcome"]               # diabetes yes/no

### Split data into learning and testing

In [33]:
# 4. Split data into learning and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Logistic Regression with scaling

In [34]:
# 5. Logistic Regression with scaling (BEST PRACTICE)
log_model = Pipeline([
    ("scaler", StandardScaler()),       
# StandardScaler transforms each column so that:
#  → mean (average) becomes 0
#  → standard deviation becomes 1

    ("model", LogisticRegression(max_iter=2000))
])

### Decision Tree

In [36]:
tree_model = DecisionTreeClassifier(
    max_depth=5,        # stop memorizing
    min_samples_split=20
)

### Train both models

In [37]:
log_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=5, min_samples_split=20)

### Test accuracy

In [40]:
log_acc = accuracy_score(y_test, log_model.predict(X_test))
tree_acc = accuracy_score(y_test, tree_model.predict(X_test))

print("Logistic Regression Accuracy:", log_acc)
print("Decision Tree Accuracy:", tree_acc)

print("\n MODEL COMPARISON:")
print(f"  Logistic Regression → Accuracy: {log_acc*100:.2f}%  |  AUC: {log_acc:.4f}")
print(f"  Decision Tree       → Accuracy: {tree_acc*100:.2f}%  |  AUC: {tree_acc:.4f}")

Logistic Regression Accuracy: 0.7662337662337663
Decision Tree Accuracy: 0.7857142857142857

 MODEL COMPARISON:
  Logistic Regression → Accuracy: 76.62%  |  AUC: 0.7662
  Decision Tree       → Accuracy: 78.57%  |  AUC: 0.7857


### Save the better model

In [41]:
best_model = log_model if log_acc > tree_acc else tree_model
joblib.dump(best_model, "model.pkl")

['model.pkl']